# 04. Python: funkcje

Czas: ok. 25 min

**Czego się nauczysz**
- funkcja to przepis wielokrotnego użytku: piszesz raz, używasz dla każdego z 456 przetargów
- parametry, czyli dane wejściowe funkcji, i (opcjonalnie) wartości domyślne
- `return` (oddaj wynik do dalszej pracy) kontra `print` (tylko pokaż na ekranie)
- jak krok po kroku zbudować `clean_amount`, która zamienia `"1 234 567,89 zł"` na liczbę
- (opcjonalnie) druga mała funkcja `has_code`: czy przetarg obejmuje dany kod odpadów

## 1. Po co funkcje: jeden przepis dla 456 przetargów

Dzielenie `wartość / wolumen` (cena za Mg) można wpisywać za każdym razem od nowa. Funkcja to przepis z nazwą:
opisujesz go raz, a potem tylko wywołujesz, podając dane konkretnego przetargu.
Zasady jak przy `if`: dwukropek na końcu linii z `def` i wcięcie 4 spacje dla wszystkiego, co należy do funkcji.

In [ ]:
# def = "definiuję funkcję": nazwa, w nawiasie parametry (dane wejściowe), dwukropek, wcięcie.
def price_per_mg(value, volume):
    return value / volume      # return = oddaj wynik temu, kto wywołał funkcję

# Wywołanie: nazwa + nawiasy z konkretnymi wartościami. Dopiero teraz kod w środku się wykonuje.
price = price_per_mg(9_500_000, 12_500)    # wynik trafia do zmiennej price; podkreślniki w liczbie są tylko dla czytelności (9_500_000 = 9 500 000)
print(price)

# Ten sam przepis dla innego przetargu: zmieniają się dane, nie kod.
price_2 = price_per_mg(24_300_000.50, 31_200)
print(round(price_2, 2))                          # round(liczba, 2) = zaokrąglij do 2 miejsc po przecinku

Pułapki (przeczytaj, nie uruchamiaj):

```python
price_per_mg(9_500_000)          # TypeError: funkcja chce dwóch wartości, dostała jedną
price_per_mg(12_500, 9_500_000)  # bez błędu, ale cena to bzdura: kolejność wartości ma znaczenie
price_per_mg                     # bez nawiasów nic się nie liczy
```

Python czyta od góry: komórkę z `def` uruchom przed komórką z wywołaniem, inaczej dostaniesz `NameError` (Python nie zna jeszcze tej nazwy).

**Twoja kolej:** policz cenę za Mg dla przetargu z Twoich danych.

In [ ]:
# TODO: podmień wartość umowy i wolumen na liczby z dowolnego przetargu (np. z Twojego Excela).
contract_value = 4_800_000       # wartość umowy brutto w zł
waste_volume = 6_000             # łączny wolumen w Mg
my_price = price_per_mg(contract_value, waste_volume)   # wywołanie: podajemy zmienne zamiast liczb
print(f"Cena za Mg: {my_price:.2f} zł")                 # f-string: :.2f = 2 miejsca po przecinku

## 2. return kontra print

`return` oddaje wynik na zewnątrz: można go zapisać do zmiennej i liczyć dalej.
`print` tylko pokazuje wynik na ekranie, a potem wynik przepada.
Typowy błąd początkującego: funkcja "działa", bo coś wypisuje, ale do zmiennej trafia `None` (nic).

In [ ]:
# Wersja z print: pokazuje wynik, ale niczego nie oddaje.
def show_price(value, volume):
    print(value / volume)

shown = show_price(9_500_000, 12_500)       # wypisze 760.0, ale...
print("W zmiennej shown jest:", shown)      # ...zmienna dostała None, czyli nic

# Wersja z return (nasza price_per_mg): wynik można zapisać i użyć dalej.
price_a = price_per_mg(9_500_000, 12_500)
price_b = price_per_mg(24_300_000.50, 31_200)
print("Różnica cen za Mg:", round(price_a - price_b, 2))   # z None takie odejmowanie dałoby błąd

## 3. Funkcja z if: poziom konkurencji

W środku funkcji działają zwykłe `if/elif/else`, porównania i `and`/`or`, dokładnie tak samo jak poza funkcją.
`return` kończy funkcję od razu: gdy pierwszy warunek jest spełniony, reszta się nie wykona.

In [ ]:
# Liczba ofert -> opis słowny: 1 oferta, 2 oferty, 3 i więcej. Każda gałąź if kończy się własnym return.
def competition_level(offers_count):
    if offers_count == 1:              # uwaga: == porównuje, = przypisuje
        return "brak konkurencji"
    elif offers_count == 2:
        return "słaba konkurencja"
    else:                              # 3 i więcej ofert
        return "konkurencja"

print(competition_level(1))
print(competition_level(2))
print(competition_level(4))

## 4. Parametr domyślny: próg dużej umowy (opcjonalnie, jeśli zostanie czas)

Parametr domyślny ma wartość "z góry", używaną, gdy nic nie podasz. W naszych danych umowy
powyżej 1 mln zł traktujemy jako duże, ale próg da się zmienić, dopisując jedną wartość przy wywołaniu.

In [ ]:
# threshold=1_000_000 to wartość domyślna: działa, gdy wywołujemy funkcję tylko z wartością umowy.
def is_large_contract(value, threshold=1_000_000):
    return value > threshold           # porównanie daje True/False; to też można zwrócić

print(is_large_contract(9_500_000))                    # próg domyślny 1 mln: True
print(is_large_contract(950_000))                      # poniżej progu: False
print(is_large_contract(950_000, 500_000))             # własny próg 500 tys.: True
print(is_large_contract(950_000, threshold=500_000))   # to samo z nazwą parametru: czytelniej

## 5. Budujemy clean_amount krok po kroku

W Excelu z przetargami kwota to tekst w wielu formatach: `"1 234 567,89 zł"`, `"987 654,50 PLN"`, `"1234567,89"`,
a Python z tekstu nic nie policzy, dopóki nie zamieni go na liczbę. Piszemy funkcję, która z każdego formatu wyciąga liczbę.
Budujemy ją w 4 krokach; każda komórka to pełna, nowa wersja funkcji z jednym dołożonym fragmentem.
⚠️ Ponowne `def clean_amount` PODMIENIA poprzednią wersję: Python pamięta tylko ostatnio uruchomioną komórkę z `def`,
więc kroki uruchamiaj po kolei (a), (b), (c), (d).

**Krok (a):** zamień wejście na tekst. Do funkcji może przyjść tekst, ale też liczba albo brak wartości (`None`),
a pętla po znakach działa tylko na tekście.

In [ ]:
# Krok (a): str() zamienia cokolwiek na tekst; tekst zostaje tekstem.
def clean_amount(text):
    return str(text)

print(clean_amount("1 234 567,89 zł"))   # tekst: bez zmian
print(clean_amount(1234567.89))          # liczba (Excel czasem tak oddaje): teraz tekst "1234567.89"
print(clean_amount(None))                # brak wartości: tekst "None" (bez cyfr; zajmiemy się nim w kroku (c))
print(type(clean_amount(None)))          # <class 'str'>: to tekst "None", nie brak wartości (type() = jaki to typ)

**Krok (b):** pętla `for char in tekst` przechodzi po tekście znak po znaku i zostawia tylko cyfry, przecinek i kropkę.
Spacje, `zł`, `PLN`, `Mg` wypadają. Zmienna `digits` żyje tylko w środku funkcji; na zewnątrz dostajesz wyłącznie to, co odda `return`.
W wynikach Python pokazuje tekst w pojedynczych cudzysłowach `'...'`; to to samo, co podwójne `"..."` w naszym kodzie.

In [ ]:
# Krok (b): zbieramy "dobre" znaki do nowego tekstu.
def clean_amount(text):                        # ta sama nazwa: ta wersja podmienia wersję z kroku (a)
    digits = ""                                # tu doklejamy cyfry, przecinek i kropkę
    for char in str(text):                     # dla każdego znaku tekstu...
        if char.isdigit() or char in ",.":     # ...czy to cyfra, przecinek albo kropka?
            digits = digits + char             # tak: doklejamy na koniec
    return digits

print(clean_amount("1 234 567,89 zł"))   # "1234567,89": spacje i "zł" wypadły
print(clean_amount("12 500,00 Mg"))      # "12500,00"
# repr() pokazuje tekst "od kuchni", w cudzysłowach: tak widać, że coś jest pustym tekstem.
print(repr(clean_amount("brak danych")))   # '' czyli pusty tekst: ani jednej cyfry

**Krok (c):** gdy nie zebraliśmy żadnej cyfry, oddajemy `None`, czyli "brak wartości". Pusty tekst `""` nie jest liczbą,
a w tabeli chcemy mieć wyraźnie: tu nic nie ma.

In [ ]:
# Krok (c): pusty wynik -> None.
def clean_amount(text):                        # znów ta sama nazwa: podmienia wersję z kroku (b)
    digits = ""
    for char in str(text):
        if char.isdigit() or char in ",.":
            digits = digits + char
    if digits == "":          # nic nie zebraliśmy: to nie była kwota
        return None           # return kończy funkcję, ostatnia linia już się nie wykona
    return digits

print(clean_amount("brak danych"))   # None
print(clean_amount(None))            # None (bo w tekście "None" nie ma cyfr); na oko jak w kroku (a)...
print(type(clean_amount(None)))      # ...ale <class 'NoneType'>: prawdziwy brak wartości, nie tekst "None"
print(clean_amount("1 234 567,89 zł"))   # "1234567,89": nadal tekst z przecinkiem, liczbę zrobi krok (d)

**Krok (d):** przecinek na kropkę i `float()` (zamiana tekstu na liczbę), bo Python rozumie tylko kropkę dziesiętną. To jest gotowa funkcja:
w notebooku 06 (Pandas: czyszczenie danych) wkleimy ją bez zmian. Tekst w potrójnych cudzysłowach pod `def` to opis funkcji (tzw. docstring),
notatka dla człowieka, Python go pomija.

In [ ]:
# Krok (d): wersja ostateczna = krok (c) + przecinek na kropkę + float(). Ostatnia podmiana funkcji.
def clean_amount(text):
    """Zamienia tekst w stylu '1 234 567,89 zł' na liczbę 1234567.89. Gdy nie ma cyfr -> None."""
    digits = ""
    for char in str(text):
        if char.isdigit() or char in ",.":
            digits = digits + char
    if digits == "":
        return None
    return float(digits.replace(",", "."))   # "1234567,89" -> "1234567.89" -> liczba 1234567.89

result = clean_amount("1 234 567,89 zł")
print(result)          # 1234567.89
print(type(result))    # float: wreszcie liczba, można ją dzielić i sumować

In [ ]:
# Test na wszystkich formatach kwot, wolumenu i okresu z pliku z przetargami (i na "śmieciach").
test_values = [
    "1 234 567,89 zł", "987 654,50 PLN", "1234567,89",   # formaty kwot z pliku
    "12 500,00 Mg", "24 miesiące", "24 mies.",            # wolumen i okres: ta sama funkcja (skrót z kropką też działa)
    "brak danych", None,                                   # "śmieci": ma wyjść None
]
for value in test_values:                        # dla każdego tekstu z listy...
    print(value, "->", clean_amount(value))      # ...pokaż wejście i wynik

**Twarda spacja `\xa0`** (opcjonalnie, jeśli zostanie czas): niewidoczny wróg ze scrapowania (automatycznego pobierania danych ze stron). Wygląda jak spacja, ale to inny znak,
więc zwykłe `replace(" ", "")`, które usuwa spacje, by jej nie tknęło. Naszej funkcji to nie rusza, bo zostawia tylko cyfry, przecinek i kropkę.

In [ ]:
# Zapis "\xa0" w kodzie to twarda spacja; w pliku z przetargami ma ją ok. 40 kwot i kilkanaście wolumenów.
sneaky_text = "1\xa0234,50 zł"
print(sneaky_text)                        # na oko: zwykłe "1 234,50 zł"
print(repr(sneaky_text))                  # repr (od kuchni): widać \xa0
print(sneaky_text == "1 234,50 zł")       # False: dla Pythona to inny tekst
print(clean_amount(sneaky_text))          # 1234.5: funkcja przepuszcza tylko cyfry, przecinek i kropkę, twarda spacja wypada

## 6. Druga funkcja: has_code (opcjonalnie, jeśli zostanie czas)

Jeśli goni czas: tylko uruchom i przeczytaj. Kody odpadów siedzą w jednym tekście, np. `"20 03 01; 15 01 01; 20 01 08"`.
`in` pyta, czy tekst zawiera dany fragment (tu: kod), i odpowiada True albo False.
Domyślnie szukamy `20 03 01` (odpady zmieszane), bo o nie pytamy najczęściej.

In [ ]:
# code="20 03 01" to wartość domyślna: gdy nie podasz kodu, funkcja sprawdza zmieszane.
def has_code(codes_text, code="20 03 01"):
    return code in codes_text          # in = czy tekst zawiera fragment; wynik True/False

waste_codes = "20 03 01; 15 01 01; 20 01 08"
print(has_code(waste_codes))                    # domyślny kod: True
print(has_code(waste_codes, "15 01 01"))        # inny kod: True
print(has_code(waste_codes, code="17 01 07"))   # tego kodu nie ma: False

## Zadania

Każde zadanie ma własną komórkę z `# TODO` i podpowiedzią. Rozwiązania są na końcu, ale najpierw spróbuj sam.

1. Napisz `contract_years(months)`, która zamienia okres umowy w miesiącach na lata.
2. Napisz `single_bid_share(offers_list)`, która pętlą liczy udział przetargów z jedną ofertą.
3. Przetestuj `clean_amount` na 3 własnych tekstach z kwotami.
4. Napisz `describe_tender(tender)`, która przyjmuje słownik przetargu i zwraca jedno zdanie (f-string).
5. Bonus: przerób `price_per_mg` tak, żeby przy braku wolumenu (`None` albo `0`) zwracała `None` zamiast błędu.

In [ ]:
# Zadanie 1: contract_years(months)
# TODO: napisz funkcję, która zwraca liczbę lat dla podanej liczby miesięcy.
# Podpowiedź: liczba lat to months / 12; oddaj ją przez return, czyli w środku funkcji: return months / 12


# Sprawdzenie (odkomentuj po napisaniu funkcji):
# contract_months = 36
# print(contract_years(contract_months))   # oczekujemy 3.0
# print(contract_years(18))                # oczekujemy 1.5

In [ ]:
# Zadanie 2: single_bid_share(offers_list)
# TODO: policz w pętli, ile przetargów ma dokładnie 1 ofertę, i zwróć udział (ile z jedną / ile wszystkich).
# Podpowiedź: single_count = 0, potem for offers_count in offers_list: i w środku if offers_count == 1:
#             single_count = single_count + 1 (to przypisanie; samo "single_count + 1" niczego nie zmienia).
#             Na końcu, już poza pętlą: return single_count / len(offers_list).


# Sprawdzenie (odkomentuj):
# offers = [1, 2, 1, 3, 1]
# print(f"Udział przetargów z jedną ofertą: {single_bid_share(offers):.1%}")   # oczekujemy 60.0% (:.1% = ułamek jako procent, 1 miejsce po przecinku)

In [ ]:
# Zadanie 3: własne testy clean_amount
# TODO: wpisz do listy 3 teksty z kwotami: np. "PLN 2 345,00" i dwa formaty, jakie masz w swoim Excelu.
# Podpowiedź: jeśli któryś format da błąd, przeczytaj komunikat: to informacja, czego funkcja "nie umie".
my_values = []                                   # <- tu wpisz swoje teksty
for value in my_values:
    print(value, "->", clean_amount(value))

In [ ]:
# Zadanie 4: describe_tender(tender)
# TODO: funkcja przyjmuje słownik (jeden przetarg = jeden wiersz tabeli, klucz = nagłówek kolumny)
#       i zwraca zdanie: "Radom: liczba ofert 2, wartość 1,250,000.50 zł".
# Podpowiedź: odczyt ze słownika to tender["gmina"]; kwotę zapisz do contract_value
#             i użyj f-string z {contract_value:,.2f} (przecinek = separator tysięcy, .2f = 2 miejsca po przecinku).
# Dodatkowo, jeśli masz czas: dopisz w nawiasie poziom konkurencji z competition_level
# (tak robi rozwiązanie na końcu).
tender = {"gmina": "Radom", "liczba_ofert": 2, "wartosc_pln": 1250000.5}


# Sprawdzenie (odkomentuj):
# print(describe_tender(tender))

In [ ]:
# Zadanie 5 (bonus): price_per_mg odporna na brak wolumenu
# TODO: przed dzieleniem sprawdź, czy volume to None albo 0; jeśli tak, zwróć None.
# Podpowiedź: if volume is None or volume == 0: return None
#             "is None" = "czy to brak wartości"; do sprawdzania None Python używa słowa is, nie ==.


# Sprawdzenie (odkomentuj):
# print(price_per_mg(9_500_000, 12_500))   # 760.0
# print(price_per_mg(9_500_000, None))     # None
# print(price_per_mg(9_500_000, 0))        # None zamiast błędu ZeroDivisionError

## Rozwiązania

In [ ]:
# Rozwiązanie 1: miesiące na lata.
def contract_years(months):
    return months / 12          # 12 miesięcy = 1 rok; dzielenie daje liczbę z przecinkiem

contract_months = 36
print(contract_years(contract_months))   # 3.0
print(contract_years(18))                # 1.5

In [ ]:
# Rozwiązanie 2: udział przetargów z jedną ofertą: licznik + pętla for + if, zamknięte w funkcję.
def single_bid_share(offers_list):
    single_count = 0                          # licznik przetargów z jedną ofertą
    for offers_count in offers_list:          # dla każdego przetargu z listy...
        if offers_count == 1:                 # ...czy była tylko jedna oferta?
            single_count = single_count + 1   # tak: licznik w górę
    return single_count / len(offers_list)    # udział = ile z jedną / ile w ogóle

offers = [1, 2, 1, 3, 1]
share = single_bid_share(offers)
print(f"Udział przetargów z jedną ofertą: {share:.1%}")   # 60.0% (:.1% = ułamek jako procent, 1 miejsce po przecinku)

In [ ]:
# Rozwiązanie 3: własne testy clean_amount na trzech nowych formatach (PLN z przodu, dopisek "brutto", spacje wokół).
my_values = ["PLN 2 345,00", "500 000 zł brutto", "  7 250,5 Mg "]
for value in my_values:
    print(value, "->", clean_amount(value))

# Pułapka: funkcja zostawia KAŻDĄ kropkę. Kropka na końcu ("24 mies.") nie szkodzi, ale kropka PRZED cyframi psuje wynik:
print("ok. 500 000 zł ->", clean_amount("ok. 500 000 zł"))   # 0.5, źle! Kropka z "ok." weszła do liczby
# "1.234.567,89" (kropka tysięcy) dałoby błąd. W naszym pliku kwoty i wolumen nie mają kropek, okres umowy tylko w "mies.";
# inne formaty (np. kropka jako separator tysięcy) wymagałyby osobnej wersji funkcji.

In [ ]:
# Rozwiązanie 4: zdanie o przetargu ze słownika (słownik = jeden wiersz tabeli, klucz = nagłówek kolumny).
def describe_tender(tender):
    municipality = tender["gmina"]              # odczyt ze słownika po kluczu
    offers_count = tender["liczba_ofert"]
    contract_value = tender["wartosc_pln"]
    level = competition_level(offers_count)     # funkcja może wywołać inną funkcję
    return f"{municipality}: liczba ofert {offers_count} ({level}), wartość {contract_value:,.2f} zł"

tenders = [
    {"gmina": "Radom", "liczba_ofert": 2, "wartosc_pln": 1250000.5},
    {"gmina": "Płock", "liczba_ofert": 1, "wartosc_pln": 18400000.0},
]
for tender in tenders:                          # ten sam przepis dla każdego przetargu z listy
    print(describe_tender(tender))

In [ ]:
# Rozwiązanie 5 (bonus): nowa wersja price_per_mg; ten def podmienia pierwszą, prostą wersję (Python pamięta ostatnią).
def price_per_mg(value, volume):
    # is None = czy brak wartości; 0 = zero: w obu przypadkach dzielenie nie ma sensu (przez 0 da błąd)
    if volume is None or volume == 0:
        return None                       # oddajemy "brak wartości" zamiast błędu
    return value / volume

print(price_per_mg(9_500_000, 12_500))   # normalnie: 760.0
print(price_per_mg(9_500_000, None))     # brak wolumenu: None
print(price_per_mg(9_500_000, 0))        # zero: None zamiast ZeroDivisionError

**Podsumowanie**

- `def nazwa(parametry):` + wcięcie + `return` to przepis; wywołanie `nazwa(wartości)` uruchamia go dla konkretnego przetargu.
- `return` oddaje wynik do dalszej pracy, `print` tylko go pokazuje. Ponowny `def` z tą samą nazwą podmienia poprzednią wersję funkcji.
- Parametr domyślny (`threshold=1_000_000`, `code="20 03 01"`) to wartość "z góry", którą możesz nadpisać przy wywołaniu.
- `clean_amount` jest gotowa: w notebooku 06 (Pandas: czyszczenie danych) wkleimy ją bez zmian, a pandas wpisze do kolumny to, co odda `return` (`.apply` dla każdej komórki).

Jeśli na zajęciach pominęliśmy części oznaczone "(opcjonalnie, jeśli zostanie czas)", przeczytaj je i uruchom w domu:
sekcję 4 (Parametr domyślny: próg dużej umowy), komórkę o twardej spacji `\xa0` w sekcji 5 (Budujemy clean_amount krok po kroku)
i sekcję 6 (Druga funkcja: has_code). Zadania 3-5 też zostają na pracę domową. Dalej: notebook 05 (Pandas: start).